# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [22]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [23]:
# TODO
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.info()

(541909, 8)
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  str           
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), f

## A.2. Missing values & Duplicate data

In [24]:
# TODO
missing = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'missing': missing, 'percent': missing_percent})
print(missing_df[missing_df['missing'] > 0].sort_values('missing', ascending=False))
print(df.duplicated().sum())

             missing    percent
CustomerID    135080  24.926694
Description     1454   0.268311
5268


## A.3. Invalid values

In [25]:
print((df['Quantity'] <= 0).sum())
print((df['UnitPrice'] <= 0).sum())
print(df['InvoiceNo'].astype(str).str.startswith('C').sum())
print(df['CustomerID'].isnull().sum())
print(df['Country'].unique())

10624
2517
9288
135080
<StringArray>
[      'United Kingdom',               'France',            'Australia',
          'Netherlands',              'Germany',               'Norway',
                 'EIRE',          'Switzerland',                'Spain',
               'Poland',             'Portugal',                'Italy',
              'Belgium',            'Lithuania',                'Japan',
              'Iceland',      'Channel Islands',              'Denmark',
               'Cyprus',               'Sweden',              'Austria',
               'Israel',              'Finland',              'Bahrain',
               'Greece',            'Hong Kong',            'Singapore',
              'Lebanon', 'United Arab Emirates',         'Saudi Arabia',
       'Czech Republic',               'Canada',          'Unspecified',
               'Brazil',                  'USA',   'European Community',
                'Malta',                  'RSA']
Length: 38, dtype: str


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [26]:
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['Sales'] = df['Quantity'] * df['UnitPrice']
df[['Quantity', 'UnitPrice', 'Sales']].head()

,Quantity,UnitPrice,Sales
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [27]:
print(df[['Quantity', 'UnitPrice', 'Sales']].mean())
print(df[['Quantity', 'UnitPrice', 'Sales']].median())
print(df[['Quantity', 'UnitPrice', 'Sales']].mode().iloc[0])

Quantity     10.542037
UnitPrice     3.907625
Sales        20.121871
dtype: float64
Quantity     3.00
UnitPrice    2.08
Sales        9.90
dtype: float64
Quantity      1.00
UnitPrice     1.25
Sales        15.00
Name: 0, dtype: float64


## Group 2 — Dispersion

In [28]:
# TODO
print(df[['Quantity', 'UnitPrice', 'Sales']].std())
print(df[['Quantity', 'UnitPrice', 'Sales']].var())
print(df['Sales'].max() - df['Sales'].min())
print(df[['Quantity', 'UnitPrice', 'Sales']].quantile(0.75) - df[['Quantity', 'UnitPrice', 'Sales']].quantile(0.25))

Quantity     155.524124
UnitPrice     35.915681
Sales        270.356743
dtype: float64
Quantity     24187.752994
UnitPrice     1289.936149
Sales        73092.768604
dtype: float64
168469.59900000002
Quantity      9.00
UnitPrice     2.88
Sales        13.95
dtype: float64


## Group 3 — Location and Shape

In [29]:
# TODO
print(df[['Quantity', 'UnitPrice', 'Sales']].quantile([0.25, 0.5, 0.75]))
print(df[['Quantity', 'UnitPrice', 'Sales']].skew())
print(df[['Quantity', 'UnitPrice', 'Sales']].kurt())

      Quantity  UnitPrice  Sales
0.25       1.0       1.25   3.75
0.50       3.0       2.08   9.90
0.75      10.0       4.13  17.70
Quantity     471.727716
UnitPrice    206.087555
Sales        506.706012
dtype: float64
Quantity     236462.342826
UnitPrice     62483.142715
Sales        297651.661046
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [30]:
# TODO
sales_by_country = df.groupby('Country')['Sales'].sum().sort_values(ascending=False)
print(sales_by_country)
print(sales_by_country.idxmax())
print(sales_by_country.max() / sales_by_country.sum() * 100)

Country
United Kingdom          9025222.084
Netherlands              285446.340
EIRE                     283453.960
Germany                  228867.140
France                   209715.110
Australia                138521.310
Spain                     61577.110
Switzerland               57089.900
Belgium                   41196.340
Sweden                    38378.330
Japan                     37416.370
Norway                    36165.440
Portugal                  33747.100
Finland                   22546.080
Singapore                 21279.290
Channel Islands           20450.440
Denmark                   18955.340
Italy                     17483.240
Hong Kong                 15691.800
Cyprus                    13590.380
Austria                   10198.680
Israel                     8135.260
Poland                     7334.650
Greece                     4760.520
Unspecified                4749.790
Iceland                    4310.000
Canada                     3666.380
USA                 

## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [31]:
sales_by_product = df.groupby('Description')['Sales'].sum().sort_values(ascending=False)
print(sales_by_product.head(10))

Description
DOTCOM POSTAGE                        206248.77
REGENCY CAKESTAND 3 TIER              174484.74
PAPER CRAFT , LITTLE BIRDIE           168469.60
WHITE HANGING HEART T-LIGHT HOLDER    106292.77
PARTY BUNTING                          99504.33
JUMBO BAG RED RETROSPOT                94340.05
MEDIUM CERAMIC TOP STORAGE JAR         81700.92
Manual                                 78112.82
POSTAGE                                78101.88
RABBIT NIGHT LIGHT                     66964.99
Name: Sales, dtype: float64


## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [32]:
# TODO
df['Month'] = df['InvoiceDate'].dt.month
print(df.groupby('Month')['Sales'].sum())

Month
1      691364.560
2      523631.890
3      717639.360
4      537808.621
5      770536.020
6      761739.900
7      719221.191
8      759138.380
9     1058590.172
10    1154979.300
11    1509496.330
12    1462538.820
Name: Sales, dtype: float64


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [33]:
# TODO
order_value = df.groupby(['Country', 'InvoiceNo'])['Sales'].sum().reset_index()
print(order_value.groupby('Country')['Sales'].mean().sort_values(ascending=False))

Country
Singapore               3039.898571
Netherlands             3036.663191
Australia               2430.198421
Japan                   1969.282632
Lebanon                 1693.880000
Hong Kong               1426.527273
Brazil                  1143.600000
Sweden                  1066.064722
Switzerland             1057.220370
Denmark                 1053.074444
Israel                  1016.907500
Norway                  1004.595556
RSA                     1002.310000
EIRE                     984.215139
Greece                   952.104000
Cyprus                   849.398750
Channel Islands          786.555385
USA                      716.078000
Spain                    684.190111
United Arab Emirates     634.093333
Iceland                  615.714286
Canada                   611.063333
Austria                  599.922353
Portugal                 581.846552
Finland                  549.904390
Malta                    545.118000
France                   534.987526
United Kingdom      

## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [34]:
# TODO
df_raw = pd.read_csv('online-retail-dataset.csv', encoding='ISO-8859-1', on_bad_lines='skip')
df_raw['is_return'] = df_raw['Quantity'] < 0
print(df_raw.groupby('Country')['is_return'].mean().sort_values(ascending=False))

Country
USA                     0.384880
Czech Republic          0.166667
Malta                   0.118110
Japan                   0.103352
Saudi Arabia            0.100000
Australia               0.058777
Italy                   0.056040
Bahrain                 0.052632
Germany                 0.047709
EIRE                    0.036847
Poland                  0.032258
Singapore               0.030568
Sweden                  0.023810
Denmark                 0.023136
Spain                   0.018950
United Kingdom          0.018552
Belgium                 0.018366
Switzerland             0.017483
France                  0.017413
European Community      0.016393
Finland                 0.014388
Hong Kong               0.013889
Channel Islands         0.013193
Norway                  0.012891
Cyprus                  0.012862
Portugal                0.011850
Austria                 0.007481
Greece                  0.006849
Israel                  0.006734
Netherlands             0.003374
Ca

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

1. United Kingdom đóng góp phần lớn doanh thu, chiếm tỷ trọng vượt trội so với các quốc gia còn lại.
2. Một số sản phẩm cụ thể chiếm phần lớn doanh thu, cho thấy doanh thu tập trung vào nhóm sản phẩm chủ lực.
3. Doanh số có xu hướng tăng vào các tháng cuối năm (mùa lễ hội), thể hiện tính mùa vụ rõ rệt.
4. Giá trị đơn hàng trung bình khác nhau đáng kể giữa các quốc gia, một số thị trường nhỏ nhưng có AOV cao hơn UK.
5. Tỷ lệ giao dịch trả hàng/hủy khác nhau giữa các quốc gia, có thể phản ánh sự khác biệt trong hành vi mua sắm hoặc chính sách đổi trả theo vùng.